In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import random
from copy import deepcopy
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms


# =============================
# Settings
# =============================

SEED = 42
DATA_ROOT = "./data"
NUM_WORKERS = 2

MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

SAVE_DIR = "/content/drive/MyDrive/ML_Project/project_files/GridSearch_Mahalanobis"
os.makedirs(SAVE_DIR, exist_ok=True)

GN_GROUPS = 32

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# =============================
# Device
# =============================

def get_best_device():

    if torch.cuda.is_available():
        return torch.device("cuda")

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")

    return torch.device("cpu")


device = get_best_device()

print(f"[INFO] Device: {device}")

PIN_MEM = device.type == "cuda"

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True


# =============================
# GroupNorm
# =============================

def make_gn(C):

    g = min(GN_GROUPS, C)

    while g > 1 and (C % g) != 0:
        g //= 2

    return nn.GroupNorm(max(1, g), C)


# =============================
# ResNet18
# =============================

def conv3x3(in_planes, out_planes, stride=1):

    return nn.Conv2d(
        in_planes,
        out_planes,
        3,
        stride,
        1,
        bias=False
    )


class BasicBlock(nn.Module):

    expansion = 1

    def __init__(self, in_planes, planes, stride=1):

        super().__init__()

        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1 = make_gn(planes)

        self.conv2 = conv3x3(planes, planes)
        self.gn2 = make_gn(planes)

        self.shortcut = nn.Sequential()

        if stride != 1 or in_planes != planes:

            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    planes,
                    1,
                    stride,
                    bias=False
                ),
                make_gn(planes)
            )


    def forward(self, x):

        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))

        out += self.shortcut(x)

        return torch.relu(out)



class ResNet18Backbone(nn.Module):

    def __init__(self, nf=64):

        super().__init__()

        self.in_planes = nf
        self.nf = nf

        self.conv1 = conv3x3(3, nf)
        self.gn1 = make_gn(nf)

        self.layer1 = self._make_layer(64, 2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)


    def _make_layer(self, planes, blocks, stride):

        layers = []

        layers.append(
            BasicBlock(self.in_planes, planes, stride)
        )

        self.in_planes = planes

        for _ in range(1, blocks):

            layers.append(
                BasicBlock(self.in_planes, planes)
            )

        return nn.Sequential(*layers)


    def forward(self, x):

        out = torch.relu(self.gn1(self.conv1(x)))

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = torch.nn.functional.avg_pool2d(out, out.size(2))

        feat = out.view(out.size(0), -1)

        return feat


    @property
    def out_dim(self):

        return 512



# =============================
# Model
# =============================

class SingleHeadNet(nn.Module):

    def __init__(self, backbone, num_classes=2):

        super().__init__()

        self.backbone = backbone
        self.head = nn.Linear(backbone.out_dim, num_classes)


    def forward(self, x):

        f = self.backbone(x)

        return self.head(f)



# =============================
# Dataset
# =============================

def get_cifar10():

    tf_train = transforms.Compose([

        transforms.RandomCrop(32, 4),
        transforms.RandomHorizontalFlip(),

        transforms.ToTensor(),

        transforms.Normalize(
            (0.4914,0.4822,0.4465),
            (0.2470,0.2435,0.2616)
        )
    ])


    tf_test = transforms.Compose([

        transforms.ToTensor(),

        transforms.Normalize(
            (0.4914,0.4822,0.4465),
            (0.2470,0.2435,0.2616)
        )
    ])


    train = datasets.CIFAR10(
        DATA_ROOT,
        True,
        download=True,
        transform=tf_train
    )


    test = datasets.CIFAR10(
        DATA_ROOT,
        False,
        download=True,
        transform=tf_test
    )


    return train, test



def filter_classes(ds, classes):

    idx = [i for i,y in enumerate(ds.targets) if y in classes]

    mapping = {c:i for i,c in enumerate(classes)}

    targets = [mapping[ds.targets[i]] for i in idx]

    sub = Subset(ds, idx)

    # store remapped labels on subset
    sub.targets = targets

    return sub



def make_loader(ds, bs, shuffle):

    return DataLoader(
        ds,
        bs,
        shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEM
    )


# =============================
# QDA Stats
# =============================

@torch.no_grad()
def compute_stats(model, loader, num_classes=2) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Compute per-class: mean, inv_cov, logdet(cov), prior (pi_k)
    returns:
      means: [K, D]
      inv_covs: [K, D, D]
      logdets: [K]  (log|cov|)
      priors: [K]  (class prior probabilities)
    """

    model.eval()

    feats = []
    labels = []

    for x,y in loader:

        x = x.to(device)

        f = model.backbone(x)

        feats.append(f.detach().cpu())
        labels.append(y.detach().cpu())

    feats = torch.cat(feats, dim=0)    # [N, D] cpu
    labels = torch.cat(labels, dim=0)  # [N]

    means = []
    inv_covs = []
    logdets = []
    priors = []

    N = feats.size(0)

    for c in range(num_classes):

        cf = feats[labels == c]  # [Nc, D]
        Nc = cf.size(0)
        if Nc == 0:
            raise RuntimeError(f"No samples found for class {c} to compute QDA stats.")

        mean = cf.mean(0)                       # [D]
        centered = cf - mean                    # [Nc, D]
        # covariance: unbiased empirical cov of features
        cov = torch.cov(centered.T)             # [D, D]
        # regularize for numerical stability
        eps = 1e-5
        cov = cov + eps * torch.eye(cov.size(0))

        # ensure positive definite: slight diagonal if slogdet non-positive
        sign, ld = torch.slogdet(cov)
        if sign.item() <= 0:
            cov = cov + 1e-3 * torch.eye(cov.size(0))
            sign, ld = torch.slogdet(cov)
            if sign.item() <= 0:
                raise RuntimeError(f"Covariance for class {c} not PD even after regularization.")

        inv = torch.inverse(cov)

        means.append(mean.to(device))
        inv_covs.append(inv.to(device))
        logdets.append(ld.to(device))
        priors.append(torch.tensor(float(Nc) / float(N), device=device))

    means = torch.stack(means, dim=0)       # [K, D]
    inv_covs = torch.stack(inv_covs, dim=0) # [K, D, D]
    logdets = torch.stack(logdets, dim=0)   # [K]
    priors = torch.stack(priors, dim=0)     # [K]

    return means, inv_covs, logdets, priors



# =============================
# QDA Evaluation
# =============================

@torch.no_grad()
def eval_qda(model, loader, means, inv_covs, logdets, priors):
    """
    QDA discriminant:
      g_k(x) = -0.5 * quad_k - 0.5 * logdet_k + log(pi_k)
    choose argmax_k g_k(x)
    """

    model.eval()

    correct = 0
    total = 0

    K = means.size(0)

    for x,y in loader:

        x = x.to(device)
        y = y.to(device)

        f = model.backbone(x)   # [B, D]

        # compute quad for each class
        # result shape: [B, K]
        quads = []
        for k in range(K):
            diff = f - means[k]                     # [B, D]
            q = torch.sum((diff @ inv_covs[k]) * diff, dim=1)  # [B]
            quads.append(q.unsqueeze(1))
        quads = torch.cat(quads, dim=1)  # [B, K]

        # logdets: [K] -> expand to [B, K]
        logdets_exp = logdets.unsqueeze(0).expand(quads.size(0), -1)  # [B, K]
        priors_log = torch.log(priors.unsqueeze(0).expand(quads.size(0), -1) + 1e-20)  # [B,K], small eps for stability

        # discriminant g_k(x)
        g = (-0.5 * quads) - 0.5 * logdets_exp + priors_log  # [B, K]

        preds = torch.argmax(g, dim=1)

        correct += (preds == y).sum().item()
        total += x.size(0)

    return correct / total

@torch.no_grad()
@torch.no_grad()
def normalize_new_classifier_weights_l2(model, n_old=0, eps=1e-12):
    w = model.head.weight  # [40, feat_dim]

    w_new = w[n_old:]  # [20, feat_dim]

    w_flat = w_new.view(w_new.size(0), -1)
    norms = w_flat.norm(2, dim=1, keepdim=True).clamp_min(eps)
    w_flat.div_(norms)
# =============================
# Training
# =============================

def train_one(model, train_loader, test_loader, epochs, lr):

    crit = nn.CrossEntropyLoss()

    opt = optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )


    best_acc = -1
    best_state = None


    for e in range(1, epochs+1):

        model.train()

        loss_sum = 0
        corr = 0
        tot = 0


        for x,y in train_loader:

            x = x.to(device)
            y = y.to(device)

            opt.zero_grad()

            out = model(x)

            loss = crit(out, y)

            loss.backward()

            opt.step()
            normalize_new_classifier_weights_l2(model, n_old=0)

            loss_sum += loss.item() * x.size(0)

            corr += (out.argmax(1)==y).sum().item()

            tot += x.size(0)


        train_acc = corr/tot
        loss_avg = loss_sum/tot


        # compute QDA stats from training data
        means, inv_covs, logdets, priors = compute_stats(
            model, train_loader, num_classes=2
        )


        # evaluate with QDA
        test_acc = eval_qda(
            model, test_loader,
            means, inv_covs, logdets, priors
        )


        if test_acc > best_acc:

            best_acc = test_acc

            best_state = deepcopy(model.state_dict())


        print(
            f"Epoch {e:03d} | "
            f"Loss {loss_avg:.4f} | "
            f"Train {train_acc*100:.2f}% | "
            f"Test(QDA) {test_acc*100:.2f}% | "
            f"Best {best_acc*100:.2f}%"
        )


    return best_acc, best_state



# =============================
# Grid Search
# =============================

def grid_search():
    train, test = get_cifar10()

    classes = [0,1]

    train = filter_classes(train, classes)
    test = filter_classes(test, classes)

    batch_sizes = [128]
    epochs_list = [200]
    lrs = [0.05]

    best_cfg = None
    best_acc = -1
    best_w = None

    init_model = SingleHeadNet(
        ResNet18Backbone(), 2
    )

    init_state = init_model.state_dict()

    for bs in batch_sizes:
        for ep in epochs_list:
            for lr in lrs:

                print(f"\n=== BS={bs} EP={ep} LR={lr} ===")

                model = SingleHeadNet(
                    ResNet18Backbone(), 2
                )

                model.load_state_dict(init_state)

                model.to(device)

                tr_loader = make_loader(train, bs, True)
                te_loader = make_loader(test, bs, False)

                acc, state = train_one(
                    model,
                    tr_loader,
                    te_loader,
                    ep,
                    lr
                )

                if acc > best_acc:
                    best_acc = acc
                    best_cfg = {
                        "bs": bs,
                        "epochs": ep,
                        "lr": lr
                    }
                    best_w = state

    print("\nBEST:", best_cfg, best_acc)

    path = os.path.join(
        SAVE_DIR,
        "best_mahalanobis_qda.pth"
    )

    torch.save({
        "state": best_w,
        "acc": best_acc,
        "cfg": best_cfg
    }, path)

    if best_w is None:
        raise RuntimeError("No best state saved (best_w is None). Check training runs.")

    print("Computing final QDA stats for best model...")

    best_model = SingleHeadNet(ResNet18Backbone(), 2).to(device)
    # best_w may be CPU tensors (from deepcopy in train), allow load regardless of device
    best_model.load_state_dict(best_w)
    best_model.eval()


    full_loader = make_loader(train, 128, False)

    means, inv_covs, logdets, priors = compute_stats(best_model, full_loader, num_classes=2)

    stats_path = os.path.join(SAVE_DIR, "best_epoch_mahalanobis_qda_stats.pth")

    torch.save({
        "means": means.cpu(),
        "inv_covs": inv_covs.cpu(),
        "logdets": logdets.cpu(),
        "priors": priors.cpu(),
        "classes": [0,1]
    }, stats_path)

    print("Saved QDA stats ->", stats_path)
    print("Saved model weights ->", path)

    return best_cfg, best_acc, path, stats_path


# =============================
# Main
# =============================

if __name__ == "__main__":

    grid_search()

Mounted at /content/drive
[INFO] Device: cuda


100%|██████████| 170M/170M [00:02<00:00, 64.4MB/s]



=== BS=128 EP=200 LR=0.05 ===
Epoch 001 | Loss 1.7383 | Train 50.63% | Test(QDA) 64.25% | Best 64.25%
Epoch 002 | Loss 0.6873 | Train 59.51% | Test(QDA) 73.10% | Best 73.10%
Epoch 003 | Loss 0.5392 | Train 73.35% | Test(QDA) 76.50% | Best 76.50%
Epoch 004 | Loss 0.5040 | Train 75.90% | Test(QDA) 79.05% | Best 79.05%
Epoch 005 | Loss 0.4706 | Train 78.09% | Test(QDA) 81.05% | Best 81.05%
Epoch 006 | Loss 0.4300 | Train 79.96% | Test(QDA) 84.15% | Best 84.15%
Epoch 007 | Loss 0.3821 | Train 83.41% | Test(QDA) 86.60% | Best 86.60%
Epoch 008 | Loss 0.3453 | Train 84.78% | Test(QDA) 86.85% | Best 86.85%
Epoch 009 | Loss 0.3000 | Train 87.32% | Test(QDA) 84.75% | Best 86.85%
Epoch 010 | Loss 0.2878 | Train 87.64% | Test(QDA) 87.35% | Best 87.35%
Epoch 011 | Loss 0.2752 | Train 88.67% | Test(QDA) 90.55% | Best 90.55%
Epoch 012 | Loss 0.2650 | Train 89.24% | Test(QDA) 88.95% | Best 90.55%
Epoch 013 | Loss 0.2599 | Train 89.41% | Test(QDA) 91.05% | Best 91.05%
Epoch 014 | Loss 0.2288 | Train 9